In [4]:
import os
import sys
import time
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../datasets/wine_preprocessed.npz"
OUT_DIR = "./outputs_wine_debug"
os.makedirs(OUT_DIR, exist_ok=True)

# DEBUG MODE: Run only 1 sample to test speed
N_TEST_SAMPLES = 1
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")
    d = np.load(npz_path, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]
    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
all_labels = sorted(np.unique(y_train).tolist())
baseline_full = np.mean(X_train, axis=0)

print("Training models...")
trained_models = {}
for m_name in ["logistic", "svm", "random_forest"]:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[j]) for j in feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

print(f"\n--- DIAGNOSTIC MODE: Running 1 Sample ---")
rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=N_TEST_SAMPLES, replace=False)

records = []

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),      # <--- watch this one
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

for model_name, model in trained_models.items():
    print(f"\nModel: {model_name}")

    for i, idx in enumerate(test_idx):
        print(f"  Sample {idx} ({i+1}/{len(test_idx)})")

        util = LocalLossUtility(
            model=model, x_full=X_test[idx], y_true=y_test[idx],
            baseline_full=baseline_full, feature_subset=np.arange(len(X_test[idx])),
            feature_names=feature_names, all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            print(f"    -> Running {method_name}...", end=" ", flush=True)
            start_time = time.time()

            try:
                # Run method
                res = method_fn(util)

                elapsed = time.time() - start_time
                print(f"Done! ({elapsed:.2f}s)")

                alloc = getattr(res, 'values', res)
                records.append({
                    "model": model_name,
                    "method": method_name,
                    "time_seconds": elapsed
                })
            except Exception as e:
                print(f"FAILED! Error: {e}")

print("\nDiagnostic complete.")

Training models...

--- DIAGNOSTIC MODE: Running 1 Sample ---

Model: logistic
  Sample 0 (1/1)
    -> Running shapley... Done! (118.31s)
    -> Running least_core... Done! (9.22s)
    -> Running gately... Done! (0.03s)
    -> Running banzhaf... Done! (118.44s)

Model: svm
  Sample 0 (1/1)
    -> Running shapley... Done! (116.51s)
    -> Running least_core... Done! (8.87s)
    -> Running gately... Done! (0.03s)
    -> Running banzhaf... Done! (117.62s)

Model: random_forest
  Sample 0 (1/1)
    -> Running shapley... Done! (446.29s)
    -> Running least_core... Done! (34.56s)
    -> Running gately... Done! (0.12s)
    -> Running banzhaf... Done! (445.36s)

Diagnostic complete.


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Iterator, Set

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# 1. Configuration
DATA_CSV_PATH = "../datasets/fao_dataset.csv"
OUT_DIR = "./outputs_fao_limiting_feature_eval2"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_SIZE = 0.2
RANDOM_STATE = 42
N_TEST_SAMPLES_FOR_EXPLANATION = 200   # set smaller/larger as needed
SAMPLE_SEED = 123

RAW_FEATURES = [
    "rainfall_mm",
    "temperature_c",
    "soil_ph",
    "soil_depth_cm",
    "drainage",
    "texture",
]

TARGET_COL = "suitability_class"
LIMITING_COL = "limiting_features"

NUM_FEATURES = ["rainfall_mm", "temperature_c", "soil_ph", "soil_depth_cm"]
CAT_FEATURES = ["drainage", "texture"]

# 2. Setup Paths & Imports for GT methods
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index

# 3. Data Loading
def load_fao_data(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found at: {csv_path}")

    print(f"Loading data from {csv_path} ...")
    df = pd.read_csv(csv_path)

    required_cols = RAW_FEATURES + [TARGET_COL, LIMITING_COL]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    X = df[RAW_FEATURES].copy()
    y = df[TARGET_COL].astype(int).copy()
    limiting = df[LIMITING_COL].astype(str).copy()

    print(f"Loaded dataframe shape: {df.shape}")
    print(f"Using raw features: {RAW_FEATURES}")
    print(f"Target distribution:\n{y.value_counts().sort_index()}")
    return df, X, y, limiting

df, X, y, limiting = load_fao_data(DATA_CSV_PATH)

N_TOTAL_SAMPLES = 1000
SUBSAMPLE_SEED = 42

if len(X) > N_TOTAL_SAMPLES:
    df_sampled = df.sample(n=N_TOTAL_SAMPLES, random_state=SUBSAMPLE_SEED)

    X = df_sampled[RAW_FEATURES].copy()
    y = df_sampled[TARGET_COL].copy()
    limiting = df_sampled[LIMITING_COL].copy()

    print(f"\nSubsampled dataset to {N_TOTAL_SAMPLES} samples")

# 4. Split + Preprocess
X_train_df, X_test_df, y_train, y_test, lim_train, lim_test = train_test_split(
    X, y, limiting,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_FEATURES),
    ],
    remainder="drop"
)

X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)

processed_feature_names = preprocessor.get_feature_names_out().tolist()
baseline_full = np.mean(X_train, axis=0)
all_labels = sorted(np.unique(y_train).tolist())

print(f"\nProcessed shapes -> X_train: {X_train.shape}, X_test: {X_test.shape}")
print("Processed feature names:")
for name in processed_feature_names:
    print("  ", name)

# 5. Map processed columns back to the 6 original features
def build_feature_group_map(processed_names: List[str]):
    """
    Maps each original base feature to the indices of transformed columns.
    Example:
        rainfall_mm -> [0]
        drainage -> [4, 5, 6, ...]  (one-hot block)
    """
    mapping = {feat: [] for feat in RAW_FEATURES}

    for idx, name in enumerate(processed_names):
        # Numeric names typically look like "num__rainfall_mm"
        # Categorical names typically look like "cat__drainage_good"
        if name.startswith("num__"):
            base = name.replace("num__", "")
            mapping[base].append(idx)

        elif name.startswith("cat__"):
            remainder = name.replace("cat__", "")
            matched = False
            for cat_feat in CAT_FEATURES:
                prefix = cat_feat + "_"
                if remainder.startswith(prefix):
                    mapping[cat_feat].append(idx)
                    matched = True
                    break
            if not matched:
                raise ValueError(f"Could not map categorical transformed column: {name}")

        else:
            raise ValueError(f"Unexpected transformed feature name: {name}")

    for feat, cols in mapping.items():
        if len(cols) == 0:
            raise ValueError(f"No processed columns mapped for base feature: {feat}")

    return mapping

feature_group_map = build_feature_group_map(processed_feature_names)

print("\nFeature group map:")
for feat, cols in feature_group_map.items():
    print(f"  {feat}: {cols}")

# 6. Train Models
trained_models = {}

MODEL_BUILDERS = {
    "logistic": lambda: LogisticRegression(
        max_iter=5000,
        multi_class="auto",
        random_state=RANDOM_STATE
    ),
    "svm": lambda: SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_STATE
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE
    ),
}

print("\nTraining models...")
for m_name, builder in MODEL_BUILDERS.items():
    model = builder()
    model.fit(X_train, y_train)
    trained_models[m_name] = model

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"\n=== {m_name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=4))

# 7. Utility class for local loss-based attributions
#    over the 6 ORIGINAL features
@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int:
        return len(self.indices)
    def __iter__(self) -> Iterator[int]:
        return iter(self.indices)
    def __getitem__(self, i: int) -> int:
        return self.indices[i]

class LocalLossUtilityGrouped:
    """
    Coalition S is a subset of ORIGINAL features (size = 6 players).
    Internally, each original feature controls one or more processed columns.
    """
    def __init__(
        self,
        model,
        x_full_processed,
        y_true,
        baseline_full_processed,
        feature_group_map,
        feature_names,
        all_labels,
    ):
        self.model = model
        self.x_full = np.asarray(x_full_processed, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full_processed, dtype=float).ravel()
        self.y_true = int(y_true)
        self.feature_group_map = feature_group_map
        self.feature_names = list(feature_names)
        self.all_labels = all_labels

        self.data = FeaturePlayers(
            indices=list(range(len(self.feature_names))),
            data_names=self.feature_names
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()

        for local_j in S:
            feat_name = self.feature_names[int(local_j)]
            cols = self.feature_group_map[feat_name]
            x[cols] = self.x_full[cols]

        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        # Higher utility = better performance = lower loss
        return -self._loss_value(self._x_for_coalition(S))

# 8. Ground-truth parsing + evaluation helpers
def parse_limiting_features(s: str) -> Set[str]:
    return set(str(s).split("|"))

def top1_hit(attr, true_set):
    ranked = np.argsort(-np.abs(attr))
    top_feature = RAW_FEATURES[int(ranked[0])]
    return int(top_feature in true_set)

def topk_recall(attr, true_set):
    k = len(true_set)
    ranked = np.argsort(-np.abs(attr))
    pred_set = {RAW_FEATURES[int(i)] for i in ranked[:k]}
    return len(pred_set & true_set) / max(len(true_set), 1)

def exact_set_match(attr, true_set):
    k = len(true_set)
    ranked = np.argsort(-np.abs(attr))
    pred_set = {RAW_FEATURES[int(i)] for i in ranked[:k]}
    return int(pred_set == true_set)

def mean_rank_of_true_features(attr, true_set):
    ranked = np.argsort(-np.abs(attr))
    rank_map = {RAW_FEATURES[int(i)]: rank + 1 for rank, i in enumerate(ranked)}
    return float(np.mean([rank_map[f] for f in true_set]))

def attribution_order(attr):
    ranked = np.argsort(-np.abs(attr))
    return [RAW_FEATURES[int(i)] for i in ranked]

# 9. Main explanation loop
rng = np.random.default_rng(SAMPLE_SEED)
n_eval = min(N_TEST_SAMPLES_FOR_EXPLANATION, len(X_test))
test_idx = rng.choice(len(X_test), size=n_eval, replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

records = []
summary_records = []

print(f"\nRunning local explanations on {len(test_idx)} test samples ...")

for model_name, model in trained_models.items():
    for method_name, method_fn in METHODS:
        print(f"\n--- Model: {model_name} | Method: {method_name} ---")

        method_rows = []

        for idx in test_idx:
            x_proc = X_test[idx]
            y_true = int(y_test.iloc[idx] if hasattr(y_test, "iloc") else y_test[idx])
            true_limiting_str = lim_test.iloc[idx] if hasattr(lim_test, "iloc") else lim_test[idx]
            true_limiting_set = parse_limiting_features(true_limiting_str)

            util = LocalLossUtilityGrouped(
                model=model,
                x_full_processed=x_proc,
                y_true=y_true,
                baseline_full_processed=baseline_full,
                feature_group_map=feature_group_map,
                feature_names=RAW_FEATURES,
                all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, "values", res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                if len(alloc) != len(RAW_FEATURES):
                    raise ValueError(
                        f"{method_name} returned {len(alloc)} values, expected {len(RAW_FEATURES)}"
                    )

                row = {
                    "model": model_name,
                    "method": method_name,
                    "test_index": int(idx),
                    "y_true": y_true,
                    "true_limiting_features": "|".join(sorted(true_limiting_set)),
                    "n_true_limiting": len(true_limiting_set),
                    "alloc_rainfall_mm": float(alloc[0]),
                    "alloc_temperature_c": float(alloc[1]),
                    "alloc_soil_ph": float(alloc[2]),
                    "alloc_soil_depth_cm": float(alloc[3]),
                    "alloc_drainage": float(alloc[4]),
                    "alloc_texture": float(alloc[5]),
                    "ranked_features_by_abs_attr": json.dumps(attribution_order(alloc)),
                    "top1_hit": top1_hit(alloc, true_limiting_set),
                    "topk_recall": topk_recall(alloc, true_limiting_set),
                    "exact_set_match": exact_set_match(alloc, true_limiting_set),
                    "mean_rank_true_features": mean_rank_of_true_features(alloc, true_limiting_set),
                }

                records.append(row)
                method_rows.append(row)

            except Exception as e:
                print(f"Error at test_index={idx} | {model_name} | {method_name}: {e}")

        if len(method_rows) > 0:
            tmp = pd.DataFrame(method_rows)
            summary_records.append({
                "model": model_name,
                "method": method_name,
                "n_samples": len(tmp),
                "top1_accuracy": tmp["top1_hit"].mean(),
                "mean_topk_recall": tmp["topk_recall"].mean(),
                "exact_set_match_accuracy": tmp["exact_set_match"].mean(),
                "mean_rank_of_true_features": tmp["mean_rank_true_features"].mean(),
            })

# 10. Save outputs
df_local = pd.DataFrame(records)
df_summary = pd.DataFrame(summary_records)

local_path = os.path.join(OUT_DIR, "local_attribution_results.csv")
summary_path = os.path.join(OUT_DIR, "limiting_feature_eval_summary.csv")

df_local.to_csv(local_path, index=False)
df_summary.to_csv(summary_path, index=False)

print("\nSaved:")
print(" ", local_path)
print(" ", summary_path)

print("\n=== Summary ===")
print(df_summary.sort_values(["model", "method"]).to_string(index=False))

Loading data from ../datasets/fao_dataset.csv ...
Loaded dataframe shape: (5000, 22)
Using raw features: ['rainfall_mm', 'temperature_c', 'soil_ph', 'soil_depth_cm', 'drainage', 'texture']
Target distribution:
suitability_class
0    1377
1    2430
2    1101
3      92
Name: count, dtype: int64

Subsampled dataset to 1000 samples

Processed shapes -> X_train: (800, 10), X_test: (200, 10)
Processed feature names:
   num__rainfall_mm
   num__temperature_c
   num__soil_ph
   num__soil_depth_cm
   cat__drainage_good
   cat__drainage_moderate
   cat__drainage_poor
   cat__texture_clay
   cat__texture_loam
   cat__texture_sandy

Feature group map:
  rainfall_mm: [0]
  temperature_c: [1]
  soil_ph: [2]
  soil_depth_cm: [3]
  drainage: [4, 5, 6]
  texture: [7, 8, 9]

Training models...

=== logistic ===
Accuracy: 0.5600
              precision    recall  f1-score   support

           0     0.5714    0.2034    0.3000        59
           1     0.6094    0.8298    0.7027        94
           2   


=== random_forest ===
Accuracy: 0.8400
              precision    recall  f1-score   support

           0     0.9375    0.5085    0.6593        59
           1     0.8288    0.9787    0.8976        94
           2     0.7963    1.0000    0.8866        43
           3     1.0000    0.7500    0.8571         4

    accuracy                         0.8400       200
   macro avg     0.8907    0.8093    0.8252       200
weighted avg     0.8573    0.8400    0.8241       200


Running local explanations on 200 test samples ...

--- Model: logistic | Method: shapley ---

--- Model: logistic | Method: least_core ---

--- Model: logistic | Method: gately ---

--- Model: logistic | Method: banzhaf ---

--- Model: svm | Method: shapley ---

--- Model: svm | Method: least_core ---

--- Model: svm | Method: gately ---

--- Model: svm | Method: banzhaf ---

--- Model: random_forest | Method: shapley ---

--- Model: random_forest | Method: least_core ---

--- Model: random_forest | Method: gately ---